# Which GPU do you need for 60 FPS in 2026? (and how fast requirements grew)

This notebook uses the [PC Game System Requirements & Estimated FPS by GPU](https://www.kaggle.com/datasets/emiliodeagustn/pc-game-system-requirements-and-estimated-fps-by-gpu) dataset: Steam system requirements for 17,400+ PC games and estimated 1080p High frame rates on 12 reference GPUs, from [PCGameBenchmarks](https://pcgamebenchmarks.com).

Three questions:
1. How much more GPU do games ask for than ten years ago?
2. Which graphics card runs most of this year's releases at 60 FPS?
3. Can *my* GPU run a given game?

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

# Works on Kaggle (/kaggle/input/...) and in the GitHub repo (../data/games.csv).
path = (glob.glob('/kaggle/input/**/games.csv', recursive=True) or glob.glob('../data/games.csv') or glob.glob('data/games.csv') or ['games.csv'])[0]
games = pd.read_csv(path)
gpus = pd.read_csv(os.path.join(os.path.dirname(path), 'gpus.csv'))
print(f'{len(games):,} games, {len(gpus):,} GPUs')
games.head(3)

## 1. System requirements over the years

`min_gpu_g3d` is the G3D benchmark score of the minimum GPU listed on Steam. The median per release year shows how quickly the floor has risen; minimum RAM follows the same path.

In [ ]:
years = games[games.release_year.between(2010, 2026)]
trend = years.groupby('release_year').agg(
    games=('name', 'size'),
    median_min_gpu_g3d=('min_gpu_g3d', 'median'),
    median_min_ram_gb=('min_ram_gb', 'median'),
)

fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax1.plot(trend.index, trend.median_min_gpu_g3d, marker='o', color='#3B82F6', label='Minimum GPU (median G3D score)')
ax1.set_ylabel('Minimum GPU, median G3D score')
ax1.set_xlabel('Release year')
ax2 = ax1.twinx()
ax2.step(trend.index, trend.median_min_ram_gb, where='mid', color='#8B5CF6', label='Minimum RAM (median GB)')
ax2.set_ylabel('Minimum RAM, median GB')
ax1.set_title('Minimum requirements of PC games by release year')
fig.legend(loc='upper left', bbox_to_anchor=(0.08, 0.88))
plt.tight_layout()
plt.show()

r = trend.median_min_gpu_g3d
print(f"Median minimum GPU score: {r.loc[2016]:.0f} in 2016 -> {r.loc[2026]:.0f} in 2026 ({r.loc[2026] / r.loc[2016]:.1f}x)")

For scale: a GeForce GTX 1650 scores about 7,900 and an RTX 4060 about 19,500 (see `gpus.csv`). The typical 2026 minimum spec now sits between a GTX 1050 (about 5,000) and a GTX 1050 Ti (about 6,400), more than ten times the 2016 figure, and it keeps climbing.

## 2. Which GPU runs this year's games at 60 FPS?

Share of games released in 2025-2026 that each reference GPU runs at 60 FPS or more at 1080p, High preset (estimates, no upscaling).

In [ ]:
recent = games[games.release_year >= 2025]
fps_cols = [c for c in games.columns if c.startswith('fps_1080p_high_')]
share = (recent[fps_cols] >= 60).mean().sort_values()
labels = [c.replace('fps_1080p_high_', '').replace('_', ' ').upper().replace('RADEON ', 'Radeon ').replace('IRIS XE', 'Iris Xe') for c in share.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels, share.values * 100, color=['#22D3EE' if v >= 0.9 else '#3B82F6' if v >= 0.6 else '#64748B' for v in share.values])
ax.bar_label(bars, fmt='%.0f%%', padding=3)
ax.set_xlim(0, 110)
ax.set_xlabel('% of 2025-2026 releases at 60+ FPS (1080p High)')
ax.set_title(f'Which GPU runs this year\'s games at 60 FPS? ({len(recent):,} games)')
plt.tight_layout()
plt.show()

Roughly: an **RTX 4060** already covers most 2025-2026 releases at 1080p High / 60 FPS, the **RTX 5060** almost all of them, and from an **RTX 4070 / RX 7800 XT** up it is practically everything. Budget cards like the GTX 1650 and GTX 1660 Super handle roughly half of this year's games at those settings, and integrated graphics far fewer: lower presets and upscaling are the way to play the rest.

The most demanding recent games by recommended GPU:

In [ ]:
cols = ['name', 'release_year', 'rec_gpu', 'rec_gpu_g3d', 'fps_1080p_high_rtx_4060', 'fps_1080p_high_rtx_4070', 'page_url']
recent.dropna(subset=['rec_gpu_g3d']).sort_values('rec_gpu_g3d', ascending=False)[cols].head(10)

## 3. Can my GPU run it?

A small helper over the dataset. For any GPU (not only the 12 reference cards), every resolution and quality preset, the link in `page_url` opens the full table on PCGameBenchmarks.

In [ ]:
def can_i_run(game_name, gpu_column='fps_1080p_high_rtx_4060'):
    rows = games[games.name.str.lower() == game_name.lower()]
    if rows.empty:
        rows = games[games.name.str.contains(game_name, case=False, regex=False)].head(3)
    for _, g in rows.iterrows():
        fps = g[gpu_column]
        verdict = 'runs great' if fps >= 60 else 'playable' if fps >= 30 else 'struggles'
        gpu = gpu_column.replace('fps_1080p_high_', '').replace('_', ' ').upper()
        print(f"{g['name']} ({int(g.release_year) if pd.notna(g.release_year) else '?'}): ~{fps:.0f} FPS on {gpu} at 1080p High -> {verdict}")
        print(f"  minimum GPU on Steam: {g.min_gpu}")
        print(f"  full table: {g.page_url}")

can_i_run('Cyberpunk 2077')
can_i_run('Elden Ring', 'fps_1080p_high_gtx_1660_super')
can_i_run('Resident Evil Requiem', 'fps_1080p_high_rtx_3060')

## Notes

- FPS figures are **estimates** from a model calibrated on ~1,800 published benchmark results (mean error about x1.4, no overall bias), not measurements. CPU, RAM, drivers and the scene all matter.
- Requirement texts are as listed on each game's Steam store page.
- Data: [PCGameBenchmarks](https://pcgamebenchmarks.com) (CC BY 4.0). New releases are added there daily.